In [1]:
import pandas as pd
import nltk

# Downloads linguistic data files
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger_eng")

from src.data.augmentation import (
    AugmentConfig,
    augment_dataframe,
    class_balanced_augment,
    TextAugmenter
)



[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\30694\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\30694\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\30694\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [2]:
df = pd.read_csv("data/emotion_processed_train.csv")
df.head()

,text,label,clean_text,clean_word_len
0,i didnt feel humiliated,0,i didnt feel humiliated,4
1,i can go from feeling so hopeless to so damned...,0,i can go from feeling so hopeless to so damned...,21
2,im grabbing a minute to post i feel greedy wrong,3,im grabbing a minute to post i feel greedy wrong,10
3,i am ever feeling nostalgic about the fireplac...,2,i am ever feeling nostalgic about the fireplac...,18
4,i am feeling grouchy,3,i am feeling grouchy,4


In [3]:
df.columns
df.shape

(16000, 4)

In [4]:
df['label'].value_counts()


label
1    5362
0    4666
3    2159
4    1937
2    1304
5     572
Name: count, dtype: int64

In [5]:
config = AugmentConfig(
    mode="eda",
    n_aug_per_sample=1,
    seed=42,
    keep_original=True
)

In [6]:
aug_df = augment_dataframe(
    df,
    text_col="clean_text",
    label_col='label',
    config=config
)

print("Original shape:", df.shape)
print("Augmented shape:", aug_df.shape)

Original shape: (16000, 4)
Augmented shape: (27858, 4)


In [7]:
bal_config = AugmentConfig(
    mode="eda",
    n_aug_per_sample=1,
    seed=42,
    min_tokens=4,
    keep_original=True
)

balanced_df = class_balanced_augment(
    df,
    text_col="clean_text",
    label_col="label",
    target_per_class=None,
    config=bal_config
)

print("Before:", df.shape)
print("After:", balanced_df.shape)

Before: (16000, 4)
After: (32172, 4)


In [8]:
after_counts = balanced_df["label"].value_counts()
print(after_counts)


label
1    5362
2    5362
0    5362
5    5362
4    5362
3    5362
Name: count, dtype: int64


In [9]:
example = df.sample(3, random_state=42)[["clean_text","label"]]
print(example)

                                             clean_text  label
8756  ive made it through a week i just feel beaten ...      0
4660                 i feel this strategy is worthwhile      1
6095  i feel so worthless and weak what does he have...      0


In [10]:

demo_config = AugmentConfig(
    mode="eda",
    n_aug_per_sample=3,
    seed=42,
    min_tokens=4
)

demo_augmenter = TextAugmenter(demo_config)

In [11]:
for i, row in example.iterrows():
    original = row["clean_text"]
    label = row["label"]

    augmented = demo_augmenter.augment_text(original)

    print("=" * 80)
    print(f"Label: {label}")
    print(f"Original:\n{original}\n")

    for j, aug in enumerate(augmented, 1):
        print(f"Augmented {j}:\n{aug}\n")


Label: 0
Original:
ive made it through a week i just feel beaten down

Augmented 1:
ive made it through a week i just feel beaten grim

Augmented 2:
ive made it through a week i just feel beaten downward

Augmented 3:
ive throw it through a week i just feel beaten down

Label: 1
Original:
i feel this strategy is worthwhile

Augmented 1:
i palpate this strategy is worthwhile

Augmented 2:
i feel this scheme is worthwhile

Label: 0
Original:
i feel so worthless and weak what does he have to say that s what i want to find out

Augmented 1:
i feel so worthless and weak what does he have to say that s what i want to find come out of the closet

